# 03 — Héritage, `super()` et MRO

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- créer une sous-classe avec `class Fille(Mere):`
- appeler le constructeur parent via `super().__init__(...)`
- surcharger une méthode et déléguer au parent
- comprendre le **Method Resolution Order** (MRO) et l'héritage multiple
- utiliser `isinstance` et `issubclass` correctement
- connaître les pièges de l'héritage multiple (diamant, `super()`)

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- classes, `__init__`, `self`
- `@property`, `@classmethod`, `@staticmethod`
- `__repr__`, `__str__`

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- surcharge de `__eq__`, `__lt__`, `__add__` (notebook 04)
- `ABC` et `Protocol` (notebook 05)
- `dataclass` et son héritage (jour 2)

## Plan

1. Héritage simple : `class B(A):`
2. `super().__init__(...)` — propager la construction
3. Surcharger une méthode
4. `isinstance` et `issubclass`
5. Héritage multiple et le MRO
6. Le diamant et pourquoi `super()` existe
7. Anti-patterns fréquents
8. Synthèse
9. Exercices

---

## 1. Héritage simple : `class B(A):`

L'héritage sert à réutiliser une classe en la **spécialisant**. Toute sous-classe commence avec les attributs et méthodes de sa classe mère.

In [ ]:
class Salle:
    def __init__(self, nom: str, capacite: int) -> None:
        self.nom = nom
        self.capacite = capacite

    def decrire(self) -> str:
        return f'Salle {self.nom} ({self.capacite} places)'


In [ ]:
class SalleReunion(Salle):
    pass  # ne définit rien, hérite tout


In [ ]:
r = SalleReunion("Mars", 12)


In [ ]:
r.decrire()


In [ ]:
isinstance(r, SalleReunion), isinstance(r, Salle)


---

## 2. `super().__init__(...)` — propager la construction

Quand la sous-classe a ses propres attributs, il faut **explicitement** appeler le constructeur parent, sinon on repart de zéro.

In [ ]:
class SalleReunion(Salle):
    def __init__(self, nom: str, capacite: int, *, visio: bool) -> None:
        super().__init__(nom, capacite)  # délègue au parent
        self.visio = visio

    def decrire(self) -> str:
        base = super().decrire()  # réutilise la méthode parente
        return f'{base} — visio : {"oui" if self.visio else "non"}'


In [ ]:
r = SalleReunion("Mars", 12, visio=True)


In [ ]:
r.decrire()


### Pourquoi `super()` plutôt que `Salle.__init__(self, ...)` ?

Parce que `super()` suit le **MRO** (voir section 5). Si un jour votre classe est insérée dans une hiérarchie plus complexe, `super()` continue à marcher ; l'appel par nom cassera l'enchaînement.

---

## 3. Surcharger une méthode

Le terme consacré est **override** (surcharge en français). On redéfinit une méthode de même nom dans la sous-classe. On peut :

- remplacer entièrement (ne pas appeler `super()`) ;
- enrichir (appeler `super()` puis compléter).

In [ ]:
class SalleFormation(Salle):
    def __init__(self, nom: str, capacite: int, nb_ordinateurs: int) -> None:
        super().__init__(nom, capacite)
        self.nb_ordinateurs = nb_ordinateurs

    def decrire(self) -> str:
        return (
            f'Salle de formation {self.nom} : {self.capacite} places, '
            f'{self.nb_ordinateurs} postes'
        )


In [ ]:
f = SalleFormation("Io", 15, 15)


In [ ]:
f.decrire()


---

## 4. `isinstance` et `issubclass`

Les tests de type **classiques** : `isinstance(obj, ClasseOuTuple)` et `issubclass(Classe, ParentOuTuple)`. Ils acceptent un tuple pour tester plusieurs possibilités d'un coup.

In [ ]:
r = SalleReunion("Mars", 12, visio=True)


In [ ]:
isinstance(r, Salle)


In [ ]:
isinstance(r, (SalleReunion, SalleFormation))


In [ ]:
issubclass(SalleReunion, Salle)


In [ ]:
issubclass(SalleReunion, SalleFormation)


### `type(x) is C` vs `isinstance(x, C)`

- `type(x) is C` ne regarde **que** la classe exacte, pas l'héritage. À éviter sauf cas rarissimes.
- `isinstance(x, C)` accepte une sous-classe → c'est ce que vous voulez 99 % du temps.

---

## 5. Héritage multiple et le MRO

Python autorise qu'une classe hérite de **plusieurs** classes. L'ordre dans lequel Python cherche une méthode est appelé **MRO** (Method Resolution Order). Il est calculé par l'algorithme **C3 linearization**.

In [ ]:
class A:
    def qui(self) -> str:
        return 'A'

class B(A):
    def qui(self) -> str:
        return 'B'

class C(A):
    def qui(self) -> str:
        return 'C'

class D(B, C):
    pass


In [ ]:
D.__mro__


In [ ]:
D().qui()  # suit le MRO : D → B → C → A → object


### Le MRO en une phrase

Python cherche une méthode dans l'ordre `D.__mro__`, de gauche à droite, et retourne **la première** trouvée. Le MRO est l'équivalent pythonique de *« dans quel ordre j'explore les parents »*.

---

## 6. Le diamant et pourquoi `super()` existe

Quand une hiérarchie forme un diamant (deux parents partagent un grand-parent), appeler `super()` correctement est crucial pour éviter de **dupliquer** un appel au grand-parent.

In [ ]:
class Base:
    def __init__(self) -> None:
        print('Base')

class Gauche(Base):
    def __init__(self) -> None:
        print('Gauche avant')
        super().__init__()
        print('Gauche après')

class Droite(Base):
    def __init__(self) -> None:
        print('Droite avant')
        super().__init__()
        print('Droite après')

class Enfant(Gauche, Droite):
    def __init__(self) -> None:
        print('Enfant avant')
        super().__init__()
        print('Enfant après')


In [ ]:
Enfant()


In [ ]:
Enfant.__mro__


Observez : `Base` n'est appelée **qu'une seule fois**. C'est parce que `super()` suit le MRO linéarisé, qui place `Base` **une seule fois** en fin de chaîne. Sans `super()`, on appellerait `Base.__init__` deux fois.

---

## 7. Anti-patterns fréquents

Quelques pièges à éviter.

### ❌ Oublier `super().__init__()`

Si la sous-classe a besoin des attributs de la classe mère, il faut l'appeler. Sinon, accéder à `self.nom` fera planter.

### ❌ Mélanger héritage et *composition*

Si la relation n'est **pas** une « est-un » (un `SalleReunion` **est** une `Salle`), préférez la **composition** : l'objet contient un autre objet en attribut. Un `Reservation` n'est pas une `Salle` : il en a une.

### ❌ Héritage multiple pour partager 2-3 méthodes

Dans ce cas, utilisez un **mixin** (classe conçue pour être mixée) ou, mieux, un **`Protocol`** (notebook 05).

---

## Synthèse

| Outil | Rôle |
|---|---|
| `class B(A):` | B hérite de A |
| `super().meth(...)` | Appelle `meth` selon le MRO |
| `isinstance(obj, C)` | Vrai si `obj` est de type `C` ou sous-classe |
| `issubclass(A, B)` | Vrai si `A` descend de `B` |
| `C.__mro__` | Liste ordonnée de l'héritage |


### Règles à retenir

1. **Toujours `super().__init__(...)`** dans une sous-classe qui surcharge `__init__`.
2. **Préférer la composition à l'héritage** quand la relation n'est pas « est-un ».
3. **Le MRO est votre ami** : quand vous doutez, affichez `C.__mro__`.
4. **`isinstance` accepte une sous-classe**, `type() is C` non. Choisissez en conséquence.
5. **L'héritage multiple est puissant et dangereux** : limitez-le aux mixins bien conçus.

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — Premier héritage *(facile)*

Écrire une classe `Animal` avec un attribut `nom` et une méthode `parler() -> str` qui renvoie `'...'`. Puis une sous-classe `Chien` qui surcharge `parler` pour renvoyer `'Wouf !'` et une sous-classe `Chat` qui renvoie `'Miaou !'`. Tester les deux.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Heritage", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
class Animal:
    def __init__(self, nom: str) -> None:
        self.nom = nom
    def parler(self) -> str:
        return '...'

class Chien(Animal):
    def parler(self) -> str:
        return 'Wouf !'

class Chat(Animal):
    def parler(self) -> str:
        return 'Miaou !'

print(Chien('Rex').parler())
print(Chat('Felix').parler())
```

</details>

### Exercice 2 — `super()` avec attributs supplémentaires *(moyen)*

Écrire une classe `Vehicule` avec `marque` et `annee`. Écrire `Voiture(Vehicule)` qui ajoute `nb_portes` et redéfinit `__repr__` en réutilisant celui du parent.

In [1]:
# Votre code ici
class Vehicule:
    def __init__(self, marque: str, annee: int) -> None:
        self.marque = marque
        self.annee = annee

    def __repr__(self) -> str:
        return f"Vehicule(marque={self.marque!r}, annee={self.annee!r})"


class Voiture(Vehicule):
    def __init__(self, marque: str, annee: int, nb_portes: int) -> None:
        super().__init__(marque, annee)
        self.nb_portes = nb_portes

    def __repr__(self) -> str:
        parent_repr = super().__repr__()
        return f"Voiture({parent_repr}, nb_portes={self.nb_portes!r})"


voiture = Voiture("Toyota", 2020, 5)
print(voiture)

Voiture(Vehicule(marque='Toyota', annee=2020), nb_portes=5)


In [2]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Heritage", exercice=2)


📝 Exercice 2 marqué comme tenté 🟢


<details>
<summary>📖 Voir la correction</summary>

```python
class Vehicule:
    def __init__(self, marque: str, annee: int) -> None:
        self.marque = marque
        self.annee = annee
    def __repr__(self) -> str:
        return f'Vehicule(marque={self.marque!r}, annee={self.annee})'

class Voiture(Vehicule):
    def __init__(self, marque: str, annee: int, nb_portes: int) -> None:
        super().__init__(marque, annee)
        self.nb_portes = nb_portes
    def __repr__(self) -> str:
        base = super().__repr__()
        return base.replace('Vehicule', 'Voiture').rstrip(')') + f', nb_portes={self.nb_portes})'

print(Voiture('Peugeot', 2024, 5))
```

</details>

### Exercice 3 — Hiérarchie de salles (fil rouge) *(moyen)*

Écrire une hiérarchie `Salle` (classe mère) → `SalleReunion` (ajoute `visio: bool`) → `SalleFormation` (ajoute `nb_ordinateurs: int`). Chaque classe surcharge une méthode `decrire() -> str` qui appelle `super().decrire()` et ajoute sa spécificité.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Heritage", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
class Salle:
    def __init__(self, nom: str, capacite: int) -> None:
        self.nom = nom
        self.capacite = capacite
    def decrire(self) -> str:
        return f'{self.nom} ({self.capacite} places)'

class SalleReunion(Salle):
    def __init__(self, nom: str, capacite: int, visio: bool) -> None:
        super().__init__(nom, capacite)
        self.visio = visio
    def decrire(self) -> str:
        base = super().decrire()
        return f'{base}, visio={self.visio}'

class SalleFormation(SalleReunion):
    def __init__(self, nom: str, capacite: int, visio: bool, nb_ordinateurs: int) -> None:
        super().__init__(nom, capacite, visio)
        self.nb_ordinateurs = nb_ordinateurs
    def decrire(self) -> str:
        return f'{super().decrire()}, {self.nb_ordinateurs} postes'

print(SalleFormation('Io', 15, True, 15).decrire())
```

</details>

### Exercice 4 — Diamant maîtrisé *(difficile)*

Construire une hiérarchie en diamant : `A` → `B(A)` et `C(A)` → `D(B, C)`. Chaque classe affiche son nom dans `__init__` **avant** et **après** l'appel à `super().__init__()`. Instancier `D()` et tracer le parcours. Vérifier que `A` n'est appelée qu'une seule fois.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Heritage", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
class A:
    def __init__(self) -> None:
        print('A in')
        print('A out')

class B(A):
    def __init__(self) -> None:
        print('B in'); super().__init__(); print('B out')

class C(A):
    def __init__(self) -> None:
        print('C in'); super().__init__(); print('C out')

class D(B, C):
    def __init__(self) -> None:
        print('D in'); super().__init__(); print('D out')

D()
print(D.__mro__)
```

</details>

### Exercice 5 — Mixin de sérialisation *(difficile)*

Écrire un **mixin** `ToDictMixin` qui expose `to_dict(self) -> dict[str, object]` en copiant tous les attributs d'instance (via `vars(self)`). Puis une classe `Produit(ToDictMixin)` avec `nom`, `prix`, et vérifier `produit.to_dict()`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Heritage", exercice=5)


<details>
<summary>📖 Voir la correction</summary>

```python
class ToDictMixin:
    def to_dict(self) -> dict[str, object]:
        return dict(vars(self))

class Produit(ToDictMixin):
    def __init__(self, nom: str, prix: float) -> None:
        self.nom = nom
        self.prix = prix

print(Produit('pain', 1.2).to_dict())
```

</details>

---

## Ressources externes

### Documentation officielle
- [`super()` — builtin](https://docs.python.org/3/library/functions.html#super)
- [MRO — *The Python 2.3 Method Resolution Order*](https://www.python.org/download/releases/2.3/mro/)

### PEPs de référence
- **PEP 3119** — *Introducing Abstract Base Classes*

### Lectures complémentaires
- Fluent Python, chap. 14 *Inheritance: for good or for ill*.